# Resource Estimation

In this chapter, we will use the `profile` subcommand to parse the pipeline state (JSON) and estimate the necessary resources. We will compare the resource usage between the four targets `Dim2`, `Dim3`, `DistributedDim2`, and `PBC`, which we generated in the previous tutorial.


In this chapter, you will learn how to:

- Pass a pipeline state JSON file as the input for the `profile` subcommand.
- Cross-compare key metrics side-by-side, including `runtime`, `gate_count`, `code_distance`, and `num_physical_qubits`.
- Analyze how these values shift when turning on PBC (Pauli-Based Computation) mode.


In [1]:
import json
import pathlib
import os
import platform

from IPython.display import Code

project_root = pathlib.Path("../../../..").resolve()
qret_path = project_root / "build" / "main"
if platform.system() == "Darwin": 
    gridsynth_path = project_root / "externals" / "bin" / "gridsynth_macos"
else:
    gridsynth_path = project_root / "externals" / "bin" / "gridsynth"

os.environ["GRIDSYNTH_PATH"] = str(gridsynth_path)
os.environ["PATH"] = str(qret_path) + os.pathsep + os.environ.get("PATH", "")

output_dir = pathlib.Path("../../tutorial-output")
output_dir.mkdir(exist_ok=True)

## 0. Prior Check

In [ ]:
!qret profile --help

<small>Note: The `profile` subcommand cannot be run on files generated by the `asm` subcommand. This is because `profile` calculates its metrics by analyzing the `parameter`, `opt`, and `metadata` fields stored within the pipeline state JSON.</small>

## 1. Generate the Pipeline States for the Four Cases
<small>Note: This was done in the previous tutorial as well.</small>

In [5]:
dim2_pipeline_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5_dim2_pipeline.yaml"
dim3_pipeline_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5_dim3_pipeline.yaml"
dist_pipeline_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5_dist_pipeline.yaml"
pbc_pipeline_path = project_root / "quration-docs" / "tutorial" / "data" / "tutorial_5_pbc_pipeline.yaml"

!qret compile --verbose --pipeline {dim2_pipeline_path}
!qret compile --verbose --pipeline {dim3_pipeline_path}
!qret compile --verbose --pipeline {dist_pipeline_path}
!qret compile --verbose --pipeline {pbc_pipeline_path}

2026-07-17 16:18:32 - INFO  - Load OpenQASM2.
2026-07-17 16:18:32 - INFO  - Build IR from OpenQASM2.
2026-07-17 16:18:32 - INFO  - Simplify IR before compiling to SC_LS_FIXED_V0.
2026-07-17 16:18:32 - INFO  - Lowering IR to the machine function of SC_LS_FIXED_V0.
2026-07-17 16:18:32 - INFO  - Run passes.
2026-07-17 16:18:32 - INFO  - Run InitCompileInfo
2026-07-17 16:18:32 - INFO  - Initialize compile information
2026-07-17 16:18:32 - INFO  - Run Mapping
2026-07-17 16:18:32 - INFO  - Run Routing
2026-07-17 16:18:32 - INFO  - Save SC_LS_FIXED_V0 pipeline state file.
2026-07-17 16:18:32 - INFO  - Saving SC_LS_FIXED_V0 pipeline state to JSON file: ../../tutorial-output/tutorial_5_dim2.json
2026-07-17 16:18:32 - INFO  - Load OpenQASM2.
2026-07-17 16:18:32 - INFO  - Build IR from OpenQASM2.
2026-07-17 16:18:32 - INFO  - Simplify IR before compiling to SC_LS_FIXED_V0.
2026-07-17 16:18:32 - INFO  - Lowering IR to the machine function of SC_LS_FIXED_V0.
2026-07-17 16:18:32 - INFO  - Run passes

## 2. Profile The Pipelines

In [6]:
!qret profile -i {output_dir / "tutorial_5_dim2.json"} -o {output_dir / "tutorial_6_dim2_profile.json"}
!qret profile -i {output_dir / "tutorial_5_dim3.json"} -o {output_dir / "tutorial_6_dim3_profile.json"}
!qret profile -i {output_dir / "tutorial_5_dist.json"} -o {output_dir / "tutorial_6_dist_profile.json"}
!qret profile -i {output_dir / "tutorial_5_pbc.json"} -o {output_dir / "tutorial_6_pbc_profile.json"}

## 3. Compare The Key Metrics In A Table

In [7]:
def summarize(path: str) -> dict:
    data = json.loads(pathlib.Path(path).read_text())
    return {
        "execution_time_sec": data.get("execution_time_sec"),
        "gate_count": data.get("gate_count"),
        "magic_state_consumption_count": data.get("magic_state_consumption_count"),
        "code_distance": data.get("code_distance"),
        "physical_qubit_count": data.get("physical_qubit_count"),
    }


for name, path in [
    ("Dim2", output_dir / "tutorial_6_dim2_profile.json"),
    ("Dim3", output_dir / "tutorial_6_dim3_profile.json"),
    ("DistributedDim2", output_dir / "tutorial_6_dist_profile.json"),
    ("PBC", output_dir / "tutorial_6_pbc_profile.json"),
]:
    print(f"[{name}]")
    for k, v in summarize(path).items():
        print(f"  {k}: {v}")
    print()

[Dim2]
  execution_time_sec: 0.000125
  gate_count: 27
  magic_state_consumption_count: 2
  code_distance: 5
  physical_qubit_count: 4800

[Dim3]
  execution_time_sec: 0.000135
  gate_count: 32
  magic_state_consumption_count: 2
  code_distance: 5
  physical_qubit_count: 4800

[DistributedDim2]
  execution_time_sec: 0.000245
  gate_count: 31
  magic_state_consumption_count: 2
  code_distance: 7
  physical_qubit_count: 38024

[PBC]
  execution_time_sec: 0.00016099999999999998
  gate_count: 50
  magic_state_consumption_count: 2
  code_distance: 7
  physical_qubit_count: 9408



## 4. Check The PBC Result JSON

In [8]:
Code(filename=output_dir / "tutorial_6_pbc_profile.json", language="json")

{"use_magic_state_cultivation":false,"magic_factory_seed_offset":0,"magic_generation_period":15,"magic_generation_success_probability":1.0,"magic_generation_maximum_stock":10000,"entanglement_generation_period":100,"entanglement_generation_maximum_stock":10,"reaction_time":1,"topology":[{"type":"plane","coord":[10,10,0],"magic_factory":[{"symbol":0,"coord":[0,0]},{"symbol":1,"coord":[0,1]},{"symbol":2,"coord":[0,2]},{"symbol":3,"coord":[0,3]}]}],"execution_time":23,"execution_time_without_topology":22,"gate_count":50,"gate_count_detail":{"ALLOCATE":12,"ALLOCATE_MAGIC_FACTORY":4,"DEALLOCATE":12,"MEAS_ZX":9,"TWIST":4,"LATTICE_SURGERY":5,"MOVE_MAGIC":2,"XOR":2},"gate_depth":5,"gate_throughput":[11,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,1,3,7,20,4,3],"gate_throughput_ave":2.347826086956522,"gate_throughput_peak":20,"reaction_count":4,"reaction_depth":1,"reaction_rate":[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,2,1,0,0,0],"reaction_rate_ave":0.17391304347826086,"reaction_rate_peak":2,"execution_time_estimation_from_reaction_count":4,"execution_time_estimation_from_reaction_depth":1,"magic_state_consumption_count":2,"magic_state_consumption_depth":1,"magic_state_consumption_rate":[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,0,0,0,0,0,0],"magic_state_consumption_rate_ave":0.08695652173913043,"magic_state_consumption_rate_peak":2,"execution_time_estimation_magic_state_consumption_count":30,"execution_time_estimation_magic_state_consumption_depth":15,"magic_factory_count":4,"entanglement_consumption_count":0,"entanglement_consumption_depth":0,"entanglement_consumption_rate":[0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0],"entanglement_consumption_rate_ave":0.0,"entanglement_consumption_rate_peak":0,"execution_time_estimation_entanglement_consumption_count":0,"execution_time_estimation_entanglement_consumption_depth":0,"entanglement_factory_count":0,"chip_cell_count":96,"chip_cell_algorithmic_qubit":[5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,6,6,6,5,2,1,0],"chip_cell_algorithmic_qubit_ave":4.608695652173913,"chip_cell_algorithmic_qubit_peak":6,"chip_cell_algorithmic_qubit_ratio":[0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.0625,0.0625,0.0625,0.052083333333333336,0.020833333333333332,0.010416666666666666,0.0],"chip_cell_algorithmic_qubit_ratio_ave":0.0480072463768116,"chip_cell_algorithmic_qubit_ratio_peak":0.0625,"chip_cell_active_qubit_area":[7,7,5,5,5,5,5,5,5,5,5,5,5,5,5,5,17,11,21,10,4,2,0],"chip_cell_active_qubit_area_ave":6.478260869565218,"chip_cell_active_qubit_area_peak":21,"chip_cell_active_qubit_area_ratio":[0.07291666666666667,0.07291666666666667,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.052083333333333336,0.17708333333333334,0.11458333333333333,0.21875,0.10416666666666667,0.041666666666666664,0.020833333333333332,0.0],"chip_cell_active_qubit_area_ratio_ave":0.06748188405797102,"chip_cell_active_qubit_area_ratio_peak":0.21875,"qubit_volume":149,"code_distance":7,"execution_time_sec":0.00016099999999999998,"physical_qubit_count":9408}

## Compare Metrics Visually With `visualize_compile_info`
If you want to compare multiple `profile_*.json` files in a visual interface, launch the `visualize_profile.py` script:

```sh
streamlit run ../../../../quration-visualizer/visualize_profile.py
```

Main tabs:
- `Overview`: Compare primary metrics and inspect differences relative to a selected `Baseline`
- `Tables`: View detailed data tables (the "Details" section will be empty if the JSON file lacks a `gate_count_detail` field)
- `Time Series`: Select specific data series and adjust the "beat" (logical clock cycle) range.
- `Topology`: Compare physical footprint layouts across different topologies.
